# Whittaker smoothing explorer

The Whittaker smoother solves $\min_z \|y - z\|^2 + \lambda \|D_d z\|^2$. It is a zero-phase low-pass filter,

$$H(\omega) = \frac{1}{1 + \lambda\,\big(2\sin(\omega/2)\big)^{2d}},$$

so $\lambda$ is really a **cutoff period** $T_c$: the period (in bins) at which the gain falls to ½.
For $\lambda < 2^{-2d}$ the gain never falls to ½ before Nyquist, and the label reports the gain at Nyquist instead.

1. Select a data file (button, or 1b without widgets). 2–4. Plot a range of $\lambda$ — step 3 has sliders and needs widgets; step 4 does not. Bottom panel: what the smoother removed, in units of the Poisson σ —
inside ±1 it is removing noise, beyond it is removing signal.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

try:                                                     # fast sparse solver, if installed
    from whittaker_eilers import WhittakerSmoother
    def smooth(y, lam, order=2):
        ws = WhittakerSmoother(lmbda=float(lam), order=int(order), data_length=len(y))
        return np.array(ws.smooth([float(v) for v in y]))
    BACKEND = "whittaker-eilers (sparse)"
except ImportError:                                      # identical result, dense NumPy
    def smooth(y, lam, order=2):
        D = np.diff(np.eye(len(y)), order, axis=0)
        return np.linalg.solve(np.eye(len(y)) + lam * D.T @ D, np.asarray(y, float))
    BACKEND = "numpy (dense)"

def cutoff_bins(lam, order=2):
    # half-power period in bins; None if the gain stays above 1/2 up to Nyquist
    s = 0.5 * lam ** (-1 / (2 * order))
    return None if s >= 1 else np.pi / np.arcsin(s)

def lam_for_cutoff(Tc, order=2):
    # exact inverse of cutoff_bins (Tc >= 2 bins)
    return (2 * np.sin(np.pi / Tc)) ** (-2 * order)

def nyquist_gain(lam, order=2):
    return 1 / (1 + lam * 4 ** order)

def rise_time(t, y):
    # 10-90 % rise of the leading edge (None if the edge is not inside the window)
    base = np.percentile(y, 5)
    yn = (y - base) / (y.max() - base)
    ipk = int(np.argmax(yn))
    def cross(level):
        below = np.flatnonzero(yn[:ipk + 1] < level)
        if below.size == 0: raise ValueError
        i = below[-1]
        return t[i] + (level - yn[i]) / (yn[i + 1] - yn[i]) * (t[i + 1] - t[i])
    try:
        return cross(0.9) - cross(0.1)
    except ValueError:
        return None

print("smoother backend:", BACKEND)

## 1 · Select the data
Text files with one column (counts) or two or more (time, counts, …). Header lines are skipped; a unit in brackets, e.g. `Time(ns)`, is picked up.

In [ ]:
DATA = None
ON_LOAD = {}                                             # later cells register their refresh here

def parse_text(text):
    header, rows = "", []
    for line in text.splitlines():
        parts = [p for p in re.split(r"[,;\t ]+", line.strip().lstrip("#%").strip()) if p]
        if not parts:
            continue
        try:
            rows.append([float(p) for p in parts])
        except ValueError:
            header = header or line
    width = max(set(map(len, rows)), key=[len(r) for r in rows].count)   # most common row length
    a = np.array([r for r in rows if len(r) == width], float)
    if width == 1:
        return np.arange(a.shape[0], dtype=float), a[:, 0], "bin"
    unit = re.search(r"\(([^)]+)\)", header)
    return a[:, 0], a[:, 1], unit.group(1) if unit else ""

def set_data(name, raw):
    global DATA
    t, y, unit = parse_text(raw.decode("utf-8", errors="replace"))
    DATA = dict(name=name, t=t, y=y, unit=unit)

def describe():
    t, y, unit = DATA["t"], DATA["y"], DATA["unit"]
    dt = float(np.median(np.diff(t)))
    print(f"{DATA['name']}: {t.size} bins, dt = {dt:.4g} {unit}, t = {t[0]:.4g} … {t[-1]:.4g} {unit}, "
          f"peak = {y.max():.6g}, total = {y.sum():.6g}")
    r = rise_time(t, y)
    if r is None:
        return
    rb = r / dt
    print(f"rise 10–90 % = {r:.4g} {unit} = {rb:.2f} bins.  Starting points:")
    print(f"  derivative / finding m :  T_c ≈ 3 × rise   ->  lambda ≈ {lam_for_cutoff(3 * rb):.3g}")
    if 0.6 * rb > 2:
        print(f"  deconvolution          :  T_c ≤ 0.6 × rise ->  lambda ≤ {lam_for_cutoff(0.6 * rb):.3g}"
              "   (≈1 % IRF bias ceiling; high-count data prefers less)")
    else:
        print("  deconvolution          :  rise is under ~3 bins — barely resolved; smooth very little")

def uploaded_file(value):
    # (name, bytes) from FileUpload.value, for ipywidgets 7 (dict) and 8 (tuple)
    if isinstance(value, dict):
        name, item = next(iter(value.items()))
        return name, bytes(item["content"])
    return value[0]["name"], bytes(value[0]["content"])

upload = widgets.FileUpload(accept=".csv,.txt,.dat,.asc", multiple=False, description="Select data")
info = widgets.Output()

def on_upload(change):
    if not change["new"]:
        return
    set_data(*uploaded_file(change["new"]))
    with info:
        info.clear_output()
        describe()
    for refresh in ON_LOAD.values():
        refresh()

upload.observe(on_upload, names="value")
display(widgets.VBox([upload, info]))

### 1b · No widgets? Load the file here instead
If the button above shows *Error rendering output item using 'jupyter-ipywidget-renderer'*, your editor is not rendering widgets
(a known VS Code issue). This opens a native file dialog instead — or pass a path. Steps 2 and 4 then work without widgets.

In [ ]:
from pathlib import Path

def pick_file():
    # native OS file dialog; needs a local kernel (not SSH / containers / WSL without a display)
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)                # otherwise it can open behind the editor
        path = filedialog.askopenfilename(filetypes=[("Data", "*.csv *.txt *.dat *.asc"), ("All files", "*.*")])
        root.destroy()
        return path or None
    except Exception as e:
        print(f"No file dialog available ({type(e).__name__}); use load_file('path/to/file.csv') instead.")
        return None

def load_file(path):
    if not path:
        print("No file selected.")
        return
    p = Path(path)
    set_data(p.name, p.read_bytes())
    describe()
    for refresh in ON_LOAD.values():
        refresh()

load_file(pick_file())           # or: load_file("LifeTimeHistogramCh1.csv")

## 2 · Plotting function
Each λ smooths the **full** trace (no edge effects from the zoom), then the chosen window is shown.

In [ ]:
def plot_smoothing(t, y, lambdas, order=2, tlim=None, log=True, unit="", title=None):
    t, y = np.asarray(t, float), np.asarray(y, float)
    dt = float(np.median(np.diff(t)))
    sel = np.ones(t.size, bool) if tlim is None else (t >= tlim[0]) & (t <= tlim[1])
    sigma = np.sqrt(np.maximum(y, 1.0))                       # Poisson standard deviation

    fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(10, 6.5),
                               height_ratios=(3, 1.6), constrained_layout=True)
    a.plot(t[sel], y[sel], ".", ms=3.5, color="0.6", label="data", zorder=0)
    b.axhspan(-1, 1, color="0.88", zorder=0)

    for lam, c in zip(lambdas, plt.cm.viridis(np.linspace(0, 0.85, len(lambdas)))):
        z = smooth(y, lam, order)
        Tc = cutoff_bins(lam, order)
        cut = (f"T_c = {Tc:.1f} bins = {Tc * dt:.3g} {unit}" if Tc is not None
               else f"T_c < 2 bins (Nyquist gain {nyquist_gain(lam, order):.2f})")
        a.plot(t[sel], z[sel], color=c, lw=1.4, label=f"λ = {lam:.3g}   {cut}")
        b.plot(t[sel], ((y - z) / sigma)[sel], color=c, lw=0.9)

    a.set_yscale("log" if log else "linear")
    a.set_ylabel("counts")
    a.legend(fontsize=8, loc="best")
    a.set_title(f"{title or ''}   (order {order})", fontsize=10)
    b.set_yscale("symlog", linthresh=3, linscale=1.5)
    lim = max(abs(v) for v in b.get_ylim())
    ticks = [v for v in (-1000, -100, -10, -3, 0, 3, 10, 100, 1000) if abs(v) <= lim]
    b.set_yticks(ticks, [f"{v:g}" for v in ticks])
    b.set_ylabel("removed / σ")
    b.set_xlabel(f"time ({unit})" if unit else "time")
    return fig

## 3 · Explore
λ values are spaced logarithmically between *λ min* and *λ max*. Zoom the time window onto the rising edge to see where smoothing starts to eat the signal.
On a log axis, a curve that dives to the bottom has gone **negative**: the smoother rings at sharp edges.

In [ ]:
slider = dict(continuous_update=False, style={"description_width": "80px"})
w_min = widgets.FloatLogSlider(value=1e-2, base=10, min=-4, max=6, step=0.25, description="λ min",
                               readout_format=".2g", **slider)
w_max = widgets.FloatLogSlider(value=1e2, base=10, min=-4, max=6, step=0.25, description="λ max",
                               readout_format=".2g", **slider)
w_n = widgets.IntSlider(value=5, min=1, max=9, description="number of λ", **slider)
w_order = widgets.IntSlider(value=2, min=1, max=4, description="order", **slider)
w_log = widgets.Checkbox(value=True, description="log scale")
w_t = widgets.FloatRangeSlider(value=(0, 1), min=0, max=1, step=0.01, description="time window",
                               readout_format=".3g", layout=widgets.Layout(width="70%"), **slider)
out = widgets.Output()

def redraw(*_):
    with out:
        out.clear_output(wait=True)
        if DATA is None:
            print("Select a data file in step 1.")
            return
        lo, hi = sorted((w_min.value, w_max.value))
        lambdas = np.geomspace(lo, hi, w_n.value) if w_n.value > 1 else [lo]
        plot_smoothing(DATA["t"], DATA["y"], lambdas, order=w_order.value, tlim=w_t.value,
                       log=w_log.value, unit=DATA["unit"], title=DATA["name"])
        plt.show()

def refresh():
    t = DATA["t"]
    lo, hi = float(t.min()), float(t.max())
    w_t.unobserve(redraw, names="value")
    w_t.max, w_t.min = max(hi, w_t.max), min(lo, w_t.min)   # widen first, so bounds stay valid
    w_t.value = (lo, hi)
    w_t.min, w_t.max = lo, hi
    w_t.step = float(np.median(np.diff(t)))
    w_t.observe(redraw, names="value")
    redraw()

for w in (w_min, w_max, w_n, w_order, w_log, w_t):
    w.observe(redraw, names="value")
ON_LOAD["explorer"] = refresh

display(widgets.VBox([widgets.HBox([w_min, w_max, w_n]), widgets.HBox([w_order, w_log]), w_t, out]))
refresh() if DATA is not None else redraw()

## 4 · Static figure
For saving or reports — same function, explicit λ values.

In [ ]:
if DATA is not None:
    fig = plot_smoothing(DATA["t"], DATA["y"], lambdas=[0.01, 0.1, 1, 10, 100],
                         unit=DATA["unit"], title=DATA["name"])
    # fig.savefig("whittaker_smoothing.png", dpi=200)
    plt.show()
else:
    print("Select a data file in step 1, then re-run this cell.")